# Chapter 1 — Dominant Accord Profiles

Publication-oriented visualisations for *Fragrance Accord Intelligence*.

This notebook reuses the source preparation and editorial design system established in Notebook 14. It adapts the position-preserving accord extraction and median-based segmentation approach used there for ordered dyads and triads.

## Figure inventory

| Figure | Title | Scope |
|---|---|---|
| 1.1 | Most Frequently Represented Accords | Top 10 accords across all recorded positions |
| 1.2 | Distribution of Dominant Accords | All valid first-listed accords |
| 1.3 | Dominant Accord Segmentation | All eligible dominant-accord categories |
| 1.4 | Standout Profiles | Every category in the segment |
| 1.5 | Hidden Favourites | Every category in the segment |
| 1.6 | Widely Explored | Every category in the segment |
| 1.7 | Quiet Profiles | Every category in the segment |

**Interpretation conventions**
- Overall prevalence counts a fragrance wherever an accord appears; categories overlap.
- The dominant accord is the first-listed, most prominent community-perceived characteristic.
- Attention is measured by mean community votes per fragrance.
- Ratings are platform-user evaluations, not objective quality measures or evidence of sales or market demand.
- Segmentation is relative to the median attention and rating values of eligible dominant-accord categories.


## 0. Setup and source data

The source table, accord parsing and global publication design system below are reused from Notebook 14. Dominant-accord extraction adapts the position-preserving approach used for dyad and triad analysis.

In [0]:
# Install the two publication-oriented libraries used in this notebook.
# Run this cell once per Databricks cluster/environment.
%pip install lets-plot great-tables
%pip install adjustText

In [0]:
# ============================================================
# NOTEBOOK 16 — CHAPTER 1 DOMINANT ACCORD VISUALIZATIONS
# ============================================================

from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import base64
import io

from great_tables import GT, style, loc
from lets_plot import (
    LetsPlot, ggplot, aes, geom_bar, geom_text, coord_flip,
    labs, theme, theme_minimal, element_text, element_line,
    element_rect, element_blank, scale_fill_manual, scale_y_continuous,
    ggsave
)

# Configure Lets-Plot for notebook rendering.
LetsPlot.setup_html()

# Source table
SOURCE_TABLE = "main.fragrance_analytics.fragrantica_perfumes"
df_perfumes = spark.table(SOURCE_TABLE)

# Parse main_accords using the same logic as Notebook 08
df_accords = df_perfumes.withColumn(
    "accords",
    F.from_json(
        F.regexp_replace(
            F.regexp_replace(
                F.col("main_accords"),
                "'",
                '"'
            ),
            r"\bNone\b",
            "null"
        ),
        "array<string>"
    )
)

# One row per fragrance–accord combination.
# Null and empty accord labels are excluded, as in Notebook 08.
df_fragrance_accords = (
    df_accords
    .select(
        "id",
        "name",
        "gender",
        "rating_value",
        "rating_count",
        F.explode("accords").alias("accord")
    )
    .withColumn("accord", F.lower(F.trim(F.col("accord"))))
    .filter(
        F.col("accord").isNotNull()
        & (F.col("accord") != "")
    )
)

print(f"Source table: {SOURCE_TABLE}")
print(f"Source records: {df_perfumes.count():,}")
print(f"Distinct fragrance IDs: {df_perfumes.select('id').distinct().count():,}")
print(f"Distinct valid accord labels: {df_fragrance_accords.select('accord').distinct().count():,}")

## 0.1 Global visualisation standards

The shared design system below is copied from Notebook 14. All figures use the established warm-ivory background, typography, palette, header hierarchy and export conventions.

In [0]:
# ============================================================
# GLOBAL VISUALISATION DESIGN SYSTEM — ALL TABLES AND CHARTS
# ============================================================

# Single source of truth for colours, typography and sizing.
BOOK_STYLE = {
    "background": "#F7F3EC",
    "surface": "#F7F3EC",
    "text": "#302824",
    "muted": "#756B64",
    "accent": "#A47C59",
    "accent_light": "#D8C3AD",
    "rule": "#E5DDD2",
    "grid": "#E9E2D8",
    "men": "#687F8C",
    "women": "#B47C83",
    "unisex": "#A58B68",
    "other": "#B9B2AC"
}

BOOK_TYPOGRAPHY = {
    "display": "DejaVu Serif",
    "body": "DejaVu Sans"
}

BOOK_FIGURE_CHART_TOP_IN = 1.45  # Fixed physical distance from figure top to chart area

# Shared horizontal-bar layout: chart body starts closer to the left edge.
BOOK_HEADER_X = 0.06
BOOK_BAR_LEFT = 0.15

def apply_book_bar_layout(fig, bottom=0.10, right=0.96):
    """Apply the same left plot margin to every horizontal bar chart."""
    fig.subplots_adjust(
        left=BOOK_BAR_LEFT,
        right=right,
        top=1 - BOOK_FIGURE_CHART_TOP_IN / fig.get_figheight(),
        bottom=bottom
    )

BOOK_SIZES = {
    "title": 22,
    "label": 12,                 # shared readable axis/category/value labels in points
    "bar_value_mm": 4.2,         # Lets-Plot geom_text uses millimetres; approximately 12 pt
    "bar_width": 0.30,           # consistent slim bar thickness across all horizontal charts
    "title_pt": 22,                # Shared figure title size in points
    "table_body_px": "15px",
    "table_title_px": "26px",
    "table_subtitle_px": "15px",
    "section_pt": 10,
    "subtitle_pt": 13,              # Readable shared figure subtitle size in points
    "category_pt": 12,              # Shared category and axis-label size in points
    "value_pt": 12,                 # Match category labels and data-label typography
    "footer_pt": 10,
    "matrix_value_pt": 24
}

plt.rcParams.update({
    "figure.facecolor": BOOK_STYLE["background"],
    "axes.facecolor": BOOK_STYLE["background"],
    "savefig.facecolor": BOOK_STYLE["background"],
    "figure.edgecolor": BOOK_STYLE["background"],
    "text.color": BOOK_STYLE["text"],
    "axes.labelcolor": BOOK_STYLE["text"],
    "xtick.color": BOOK_STYLE["text"],
    "ytick.color": BOOK_STYLE["text"],
    "font.family": BOOK_TYPOGRAPHY["body"],
    "font.size": BOOK_SIZES["label"],
    "axes.titlesize": BOOK_SIZES["title"],
    "axes.titleweight": "bold",
    "axes.titlelocation": "center",
    "axes.labelsize": BOOK_SIZES["label"],
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

def make_book_table(data, title, subtitle=None):
    """Build a table using the global ivory, typography and title rules."""
    table = (
        GT(data)
        .tab_header(title=title, subtitle=subtitle)
        .tab_options(
            table_background_color=BOOK_STYLE["background"],
            heading_background_color=BOOK_STYLE["background"],
            column_labels_background_color=BOOK_STYLE["background"],
            table_font_names=[BOOK_TYPOGRAPHY["body"], "Arial", "sans-serif"],
            table_font_size=BOOK_SIZES["table_body_px"],
            table_font_color=BOOK_STYLE["text"],
            heading_align="left",
            heading_title_font_size=BOOK_SIZES["table_title_px"],
            heading_subtitle_font_size=BOOK_SIZES["table_subtitle_px"],
            heading_padding="14px",
            column_labels_font_weight="bold",
            column_labels_padding="10px",
            table_body_hlines_color=BOOK_STYLE["grid"],
            table_body_hlines_width="1px",
            data_row_padding="11px"
        )
    )
    table = table.tab_style(
        style.fill(color=BOOK_STYLE["background"]),
        locations=[loc.body(), loc.column_header(), loc.header()]
    )
    table = table.tab_style(
        style.text(font=BOOK_TYPOGRAPHY["display"], size=BOOK_SIZES["table_title_px"], weight="bold", color=BOOK_STYLE["text"]),
        locations=[loc.title()]
    )
    table = table.tab_style(
        style.text(font=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["table_subtitle_px"], color=BOOK_STYLE["muted"]),
        locations=[loc.subtitle()]
    )
    return table

def add_book_figure_header(fig, title, subtitle, section="FRAGRANCE INTELLIGENCE  /  CHAPTER 1"):
    """Apply identical header coordinates, typography and physical spacing to every figure."""
    figure_height = fig.get_figheight()
    section_y = 1 - 0.203 / figure_height
    title_y = 1 - 0.435 / figure_height
    subtitle_y = 1 - 0.841 / figure_height

    fig.text(
        BOOK_HEADER_X, section_y, section,
        color=BOOK_STYLE["accent"],
        fontsize=BOOK_SIZES["section_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, title_y, title,
        color=BOOK_STYLE["text"],
        fontsize=BOOK_SIZES["title_pt"],
        fontfamily=BOOK_TYPOGRAPHY["display"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, subtitle_y, subtitle,
        color=BOOK_STYLE["muted"],
        fontsize=BOOK_SIZES["subtitle_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        ha="left",
        va="top"
    )

def show_book_table(table):
    html = table.as_raw_html(all_important=True)
    displayHTML(f'<div class="nodarkmode">{html}</div>')

def show_book_plot(plot):
    html = plot.to_html(iframe=True)
    displayHTML(f'<div class="nodarkmode">{html}</div>')

def show_book_matplotlib(fig):
    buffer = io.BytesIO()
    # Keep the full figure canvas so titles retain the same visible left position
    # across figures with different tick-label widths.
    fig.savefig(
        buffer, format="png", dpi=240, bbox_inches=None,
        facecolor=BOOK_STYLE["background"],
        edgecolor=BOOK_STYLE["background"]
    )
    image_b64 = base64.b64encode(buffer.getvalue()).decode("ascii")
    displayHTML(
        '<div class="nodarkmode">'
        f'<img src="data:image/png;base64,{image_b64}" '
        'style="display:block;max-width:100%;height:auto;'
        f'background:{BOOK_STYLE["background"]};">'
        '</div>'
    )

BOOK_PLOT_THEME = (
    theme_minimal()
    + theme(
        plot_title=element_text(
            family=BOOK_TYPOGRAPHY["display"],
            size=BOOK_SIZES["title_pt"],
            face="bold",
            color=BOOK_STYLE["text"],
            hjust=0.5
        ),
        plot_subtitle=element_text(
            family=BOOK_TYPOGRAPHY["body"],
            size=BOOK_SIZES["subtitle_pt"],
            color=BOOK_STYLE["muted"],
            hjust=0.5
        ),
        axis_title=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["muted"]
        ),
        axis_text=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_text_y=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_ticks=element_blank(),
        panel_grid_major_y=element_blank(),
        panel_grid_major_x=element_line(color=BOOK_STYLE["grid"], size=0.6),
        panel_grid_minor=element_blank(),
        panel_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        plot_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        legend_position="none"
    )
)

print("Global visualisation standards are ready.")

## 1.1 Figure 1.1 — Most Frequently Represented Accords

Top ten accords across all recorded positions. The prevalence denominator is the number of fragrances with at least one valid accord. Accord categories overlap.

In [0]:
# ============================================================
# FIGURE 1.1 — MOST FREQUENTLY REPRESENTED ACCORDS
# Reuses Notebook 14's all-position prevalence logic and styling.
# ============================================================

fragrance_accords_unique_ch1 = df_fragrance_accords.dropDuplicates(["id", "accord"])
valid_accord_count_ch1 = fragrance_accords_unique_ch1.select("id").distinct().count()

accord_prevalence_ch1 = (
    fragrance_accords_unique_ch1
    .groupBy("accord")
    .agg(F.countDistinct("id").alias("fragrance_count"))
    .withColumn(
        "prevalence_pct",
        F.round(F.col("fragrance_count") / F.lit(valid_accord_count_ch1) * 100, 2)
    )
)

top_20_accords_ch1 = (
    accord_prevalence_ch1
    .orderBy(F.desc("fragrance_count"), F.asc("accord"))
    .limit(20)
)

accord_plot_ch1 = top_20_accords_ch1.toPandas().sort_values(
    "fragrance_count", ascending=True
).copy()

accord_plot_ch1["accord_label"] = accord_plot_ch1["accord"].str.title()
accord_plot_ch1["display_label"] = accord_plot_ch1.apply(
    lambda row: f"{int(row['fragrance_count']):,}  |  {row['prevalence_pct']:.2f}%",
    axis=1
)

# ------------------------------------------------------------
# Chart
# ------------------------------------------------------------

fig_1_1, ax = plt.subplots(figsize=(12, 10), dpi=180)
fig_1_1.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])

bars = ax.barh(
    accord_plot_ch1["accord_label"],
    accord_plot_ch1["fragrance_count"],
    height=0.462,  # 10% wider than the previous 0.42
    color=BOOK_STYLE["accent"],
    edgecolor="none",
    zorder=3
)

max_value = float(accord_plot_ch1["fragrance_count"].max())
ax.set_xlim(0, max_value * 1.36)

# Reduce the vertical category spacing by 20%.
# Compress category positions and bar locations together so labels remain aligned.
y_positions = np.arange(len(accord_plot_ch1)) * 0.80
ax.set_yticks(y_positions)
ax.set_yticklabels(accord_plot_ch1["accord_label"])

for bar, y, label in zip(
    bars,
    y_positions,
    accord_plot_ch1["display_label"]
):
    bar.set_y(y - bar.get_height() / 2)

    ax.text(
        bar.get_width() + max_value * 0.022,
        y,
        label,
        va="center",
        ha="left",
        fontsize=BOOK_SIZES["value_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        color=BOOK_STYLE["text"]
    )

# Set limits around the compressed category positions.
ax.set_ylim(
    y_positions[0] - 0.5,
    y_positions[-1] + 0.5
)

add_book_figure_header(
    fig_1_1,
    "Most Frequently Represented Fragrance Accords",
    "Top 20 accords across all recorded positions  |  Labels show count and catalogue prevalence."
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))

ax.tick_params(
    axis="y",
    length=0,
    labelsize=BOOK_SIZES["category_pt"],
    pad=10,
    colors=BOOK_STYLE["text"]
)

ax.tick_params(
    axis="x",
    length=0,
    labelsize=BOOK_SIZES["footer_pt"],
    pad=8,
    colors=BOOK_STYLE["muted"]
)

for tick in ax.get_yticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])
    tick.set_fontsize(BOOK_SIZES["category_pt"])
    tick.set_color(BOOK_STYLE["text"])

for tick in ax.get_xticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])

ax.grid(
    axis="x",
    color=BOOK_STYLE["grid"],
    linewidth=0.8,
    alpha=0.9,
    zorder=0
)

ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

# ------------------------------------------------------------
# Footer and layout
# ------------------------------------------------------------

fig_1_1.text(
    0.5,
    0.025,
    f"Prevalence denominator: {valid_accord_count_ch1:,} fragrances with at least one valid accord. Accords may overlap.",
    ha="center",
    fontsize=BOOK_SIZES["footer_pt"],
    fontfamily=BOOK_TYPOGRAPHY["body"],
    color=BOOK_STYLE["muted"]
)

apply_book_bar_layout(fig_1_1, bottom=0.10, right=0.85)

# Tighten the gap between the header and the chart body.
ax_pos = ax.get_position()
ax.set_position([
    ax_pos.x0,
    ax_pos.y0,
    ax_pos.width,
    ax_pos.height + 0.035
])

show_book_matplotlib(fig_1_1)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print(f"Accord prevalence denominator: {valid_accord_count_ch1:,}")
print(f"Distinct normalized accord labels: {accord_prevalence_ch1.count():,}")
print(f"Accords displayed: {len(accord_plot_ch1):,}")

## 1.2 Dominant-accord data preparation

The dominant accord is extracted from the first position of the source accord array. The all-record population is used for the distribution figure. Attention and rating aggregation uses rated records with a non-negative community rating count, following the metric eligibility logic used for dyads and triads in Notebook 14.

In [0]:
# ============================================================
# CHAPTER 1 — DOMINANT ACCORD EXTRACTION AND AGGREGATION
# Adapts Notebook 14's position-preserving dyad/triad preparation.
# ============================================================


# ------------------------------------------------------------
# 1. OUTPUT DIRECTORY
# ------------------------------------------------------------

from pathlib import Path

CHAPTER_1_OUTPUT_DIR = Path(
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)

CHAPTER_1_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. VALIDATE SOURCE ACCORD DATA
# ------------------------------------------------------------
# Apply Notebook 14's source-field validity rule before
# extracting the first accord position.

accords_clean_ch1 = F.trim(F.col("main_accords"))

valid_accords_flag_ch1 = (
    F.col("main_accords").isNotNull()
    & (accords_clean_ch1 != "")
    & (
        ~F.lower(accords_clean_ch1).isin(
            "null", "none", "nan", "n/a", "[]"
        )
    )
)


# ------------------------------------------------------------
# 3. EXTRACT THE DOMINANT ACCORD
# ------------------------------------------------------------
# The first element in the parsed accord array represents
# the dominant accord for each fragrance.

dominant_accords_all_ch1 = (
    df_accords
    .filter(valid_accords_flag_ch1)
    .select(
        "id",
        "name",
        "gender",
        "rating_value",
        "rating_count",
        F.lower(
            F.trim(F.col("accords")[0])
        ).alias("dominant_accord")
    )
    .filter(
        F.col("id").isNotNull()
        & F.col("dominant_accord").isNotNull()
        & (F.col("dominant_accord") != "")
    )
)


# ------------------------------------------------------------
# 4. VALIDATE SOURCE RECORD UNIQUENESS
# ------------------------------------------------------------
# Confirm one source row per fragrance before aggregation.
# Duplicate records would distort vote sums and mean metrics.

dominant_source_rows_ch1 = dominant_accords_all_ch1.count()

dominant_source_ids_ch1 = (
    dominant_accords_all_ch1
    .select("id")
    .distinct()
    .count()
)

assert dominant_source_rows_ch1 == dominant_source_ids_ch1, (
    f"Duplicate fragrance records detected in the dominant-accord "
    f"population: {dominant_source_rows_ch1:,} rows versus "
    f"{dominant_source_ids_ch1:,} unique IDs."
)

print(
    "Dominant-accord source validation passed: "
    f"{dominant_source_rows_ch1:,} rows / "
    f"{dominant_source_ids_ch1:,} unique IDs."
)


# ------------------------------------------------------------
# 5. DOMINANT ACCORD DISTRIBUTION
# ------------------------------------------------------------
# Include all valid fragrance records, including those
# without rating information.

dominant_accord_distribution_ch1 = (
    dominant_accords_all_ch1
    .groupBy("dominant_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
    .orderBy(
        F.desc("fragrance_count"),
        F.asc("dominant_accord")
    )
)


# ------------------------------------------------------------
# 6. DOMINANT ACCORD ATTENTION AND RATING METRICS
# ------------------------------------------------------------
# Restrict the analytical population to records with both
# rating and rating-count information.
#
# Attention: rating_count
# Evaluation: rating_value

dominant_accord_stats_ch1 = (
    dominant_accords_all_ch1
    .filter(
        F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .groupBy("dominant_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),

        F.sum("rating_count").alias("total_votes"),
        F.avg("rating_count").alias("mean_votes"),
        F.expr(
            "percentile_approx(rating_count, 0.5)"
        ).alias("median_votes"),

        F.avg("rating_value").alias("mean_rating"),
        F.expr(
            "percentile_approx(rating_value, 0.5)"
        ).alias("median_rating"),

        F.avg(
            F.when(
                F.col("rating_value") >= 4.5, 1.0
            ).otherwise(0.0)
        ).alias("share_rated_4_5_plus")
    )
)


# ------------------------------------------------------------
# 7. APPLY MINIMUM SUPPORT THRESHOLD
# ------------------------------------------------------------
# The 100-fragrance minimum is used for dominant-accord
# segmentation to avoid interpreting categories with
# insufficient representation.

MIN_DOMINANT_FRAGRANCES_CH1 = 100

eligible_dominant_accords_ch1 = (
    dominant_accord_stats_ch1
    .filter(
        F.col("fragrance_count") >= MIN_DOMINANT_FRAGRANCES_CH1
    )
)


# ------------------------------------------------------------
# 8. CONVERT ELIGIBLE RESULTS TO PANDAS
# ------------------------------------------------------------

dominant_accord_pdf_ch1 = (
    eligible_dominant_accords_ch1
    .toPandas()
    .copy()
)

numeric_metrics_ch1 = [
    "fragrance_count",
    "total_votes",
    "mean_votes",
    "mean_rating",
    "share_rated_4_5_plus"
]

for metric in numeric_metrics_ch1:
    dominant_accord_pdf_ch1[metric] = pd.to_numeric(
        dominant_accord_pdf_ch1[metric],
        errors="coerce"
    )

dominant_accord_pdf_ch1 = (
    dominant_accord_pdf_ch1
    .dropna(
        subset=[
            "dominant_accord",
            "fragrance_count",
            "mean_votes",
            "mean_rating"
        ]
    )
    .copy()
)


# ------------------------------------------------------------
# 9. VALIDATION SUMMARY
# ------------------------------------------------------------

total_dominant_categories_ch1 = (
    dominant_accord_distribution_ch1.count()
)

eligible_category_count_ch1 = len(
    dominant_accord_pdf_ch1
)

print(
    "Dominant-accord categories across all valid records: "
    f"{total_dominant_categories_ch1:,}"
)

print(
    "Categories eligible for segmentation "
    f"(n >= {MIN_DOMINANT_FRAGRANCES_CH1}): "
    f"{eligible_category_count_ch1:,}"
)


# ------------------------------------------------------------
# 10. DISPLAY DOMINANT ACCORD DISTRIBUTION
# ------------------------------------------------------------
# Display the 20 most frequently represented categories.

display(
    dominant_accord_distribution_ch1
    .select(
        F.initcap("dominant_accord").alias("dominant_accord"),
        "fragrance_count"
    )
    .limit(20)
)

## 1.3 Figure 1.2 — Distribution of Dominant Accords

The figure shows the number of fragrances assigned to each first-listed accord. Unlike overall accord prevalence, each fragrance contributes to one dominant-accord category.

In [0]:
# ============================================================
# FIGURE 1.2 — DISTRIBUTION OF DOMINANT ACCORDS
# Top 20 first-listed accord categories.
# Labels show fragrance count and percentage of total.
# ============================================================

# Total number of fragrances across all dominant accord categories.
total_dominant_fragrances_ch1 = (
    dominant_accord_distribution_ch1
    .agg(F.sum("fragrance_count").alias("total"))
    .first()["total"]
)

dominant_distribution_pdf_ch1 = (
    dominant_accord_distribution_ch1
    .orderBy(F.desc("fragrance_count"), F.asc("dominant_accord"))
    .limit(20)
    .toPandas()
    .copy()
)

dominant_distribution_pdf_ch1["dominant_accord_label"] = (
    dominant_distribution_pdf_ch1["dominant_accord"].str.title()
)

dominant_distribution_pdf_ch1["share_pct"] = (
    dominant_distribution_pdf_ch1["fragrance_count"]
    / total_dominant_fragrances_ch1
    * 100
)

dominant_distribution_pdf_ch1["display_label"] = (
    dominant_distribution_pdf_ch1.apply(
        lambda row: (
            f"{int(row['fragrance_count']):,}  |  "
            f"{row['share_pct']:.2f}%"
        ),
        axis=1
    )
)

dominant_distribution_pdf_ch1 = (
    dominant_distribution_pdf_ch1
    .sort_values("fragrance_count", ascending=True)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Chart
# ------------------------------------------------------------

fig_1_2, ax = plt.subplots(figsize=(12, 10), dpi=180)
fig_1_2.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])

# Compress category positions to 80% of their original spacing.
y_positions = np.arange(len(dominant_distribution_pdf_ch1)) * 0.80

bars = ax.barh(
    y_positions,
    dominant_distribution_pdf_ch1["fragrance_count"],
    height=0.462,
    color=BOOK_STYLE["accent"],
    edgecolor="none",
    zorder=3
)

max_value = float(dominant_distribution_pdf_ch1["fragrance_count"].max())
ax.set_xlim(0, max_value * 1.36)

ax.set_yticks(y_positions)
ax.set_yticklabels(
    dominant_distribution_pdf_ch1["dominant_accord_label"]
)

for bar, label in zip(
    bars,
    dominant_distribution_pdf_ch1["display_label"]
):
    ax.text(
        bar.get_width() + max_value * 0.018,
        bar.get_y() + bar.get_height() / 2,
        label,
        va="center",
        ha="left",
        fontsize=BOOK_SIZES["value_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        color=BOOK_STYLE["text"]
    )

ax.set_ylim(
    y_positions[0] - 0.5,
    y_positions[-1] + 0.5
)

add_book_figure_header(
    fig_1_2,
    "Distribution of Dominant Accords",
    "Top 20 first-listed accord categories  |  Labels show count and share of all dominant-accord records."
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))

ax.tick_params(
    axis="y",
    length=0,
    labelsize=BOOK_SIZES["category_pt"],
    pad=10,
    colors=BOOK_STYLE["text"]
)

ax.tick_params(
    axis="x",
    length=0,
    labelsize=BOOK_SIZES["footer_pt"],
    pad=8,
    colors=BOOK_STYLE["muted"]
)

for tick in ax.get_yticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])
    tick.set_fontsize(BOOK_SIZES["category_pt"])
    tick.set_color(BOOK_STYLE["text"])

for tick in ax.get_xticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])

ax.grid(
    axis="x",
    color=BOOK_STYLE["grid"],
    linewidth=0.8,
    alpha=0.9,
    zorder=0
)

ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

apply_book_bar_layout(fig_1_2, bottom=0.10, right=0.85)

# Reduce the gap between the header and chart body.
ax_pos = ax.get_position()
ax.set_position([
    ax_pos.x0,
    ax_pos.y0,
    ax_pos.width,
    ax_pos.height + 0.035
])

show_book_matplotlib(fig_1_2)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print(f"Total dominant-accord records: {total_dominant_fragrances_ch1:,}")
print(f"Dominant accord categories displayed: {len(dominant_distribution_pdf_ch1):,}")
print(
    f"Fragrances represented in top 20: "
    f"{dominant_distribution_pdf_ch1['fragrance_count'].sum():,}"
)
print(
    f"Combined share of top 20: "
    f"{dominant_distribution_pdf_ch1['share_pct'].sum():.2f}%"
)

## 1.4 Dominant-accord segmentation

The segmentation uses the median of mean community votes per fragrance and the median of mean fragrance ratings across eligible dominant-accord categories. As in Notebook 14's dyad and triad analysis, values equal to the median are assigned to the above-median side of the relevant dimension.

In [0]:
# ============================================================
# CHAPTER 1 — DOMINANT ACCORD SEGMENTATION
# Uses the same median classification and segment names as Notebook 14.
# ============================================================

popularity_median_dominant_ch1 = dominant_accord_pdf_ch1["mean_votes"].median()
rating_median_dominant_ch1 = dominant_accord_pdf_ch1["mean_rating"].median()

def assign_dominant_segment_ch1(row):
    high_attention = row["mean_votes"] >= popularity_median_dominant_ch1
    high_rating = row["mean_rating"] >= rating_median_dominant_ch1
    if high_attention and high_rating:
        return "Standout Profiles"
    if not high_attention and high_rating:
        return "Hidden Favourites"
    if high_attention and not high_rating:
        return "Widely Explored"
    return "Quiet Profiles"

dominant_accord_pdf_ch1["segment"] = dominant_accord_pdf_ch1.apply(
    assign_dominant_segment_ch1, axis=1
)

segment_order_dominant_ch1 = [
    "Standout Profiles", "Hidden Favourites", "Widely Explored", "Quiet Profiles"
]
segment_colors_dominant_ch1 = {
    "Standout Profiles": BOOK_STYLE["accent"],
    "Hidden Favourites": BOOK_STYLE["women"],
    "Widely Explored": BOOK_STYLE["men"],
    "Quiet Profiles": BOOK_STYLE["unisex"]
}

dominant_segment_summary_ch1 = (
    dominant_accord_pdf_ch1
    .groupby("segment", observed=False)
    .agg(
        accord_count=("dominant_accord", "count"),
        fragrance_assignments=("fragrance_count", "sum"),
        mean_votes_per_accord=("mean_votes", "mean"),
        mean_rating_per_accord=("mean_rating", "mean")
    )
    .reindex(segment_order_dominant_ch1)
    .reset_index()
)

print(f"Eligible dominant-accord categories: {len(dominant_accord_pdf_ch1):,}")
print(f"Median mean votes per fragrance: {popularity_median_dominant_ch1:,.2f}")
print(f"Median mean fragrance rating: {rating_median_dominant_ch1:.4f}")
display(dominant_segment_summary_ch1)

## 1.5 Figure 1.3 — Dominant Accord Segmentation

The bubble chart positions every eligible dominant-accord category by mean community votes per fragrance and mean fragrance rating. Bubble size represents the number of fragrances. Only selected categories are labelled for readability; the four segment charts that follow include every eligible category.

In [0]:
# ============================================================
# FIGURE 1.3 — DOMINANT ACCORD ATTENTION / RATING SEGMENTATION
# All eligible categories are plotted; selected categories are labelled.
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS AND CONFIGURATION
# ------------------------------------------------------------

from matplotlib.lines import Line2D
from adjustText import adjust_text

N_LARGEST_DOMINANT_CH1 = 8
N_EXTREMES_DOMINANT_CH1 = 6
MAX_LABELS_DOMINANT_CH1 = 24

MIN_BUBBLE_DOMINANT_CH1 = 55
MAX_BUBBLE_DOMINANT_CH1 = 950


# ------------------------------------------------------------
# 2. CALCULATE BUBBLE SIZES
# ------------------------------------------------------------
# Bubble size represents the number of fragrances in each
# dominant-accord category.
#
# Square-root scaling reduces the visual dominance of
# categories with very large fragrance counts.

dominant_counts_ch1 = (
    dominant_accord_pdf_ch1["fragrance_count"].astype(float)
)

dominant_count_min_ch1 = dominant_counts_ch1.min()
dominant_count_max_ch1 = dominant_counts_ch1.max()


def dominant_bubble_size_ch1(count):

    if dominant_count_max_ch1 == dominant_count_min_ch1:
        return MIN_BUBBLE_DOMINANT_CH1

    normalized = (
        (count - dominant_count_min_ch1)
        / (dominant_count_max_ch1 - dominant_count_min_ch1)
    )

    normalized = max(0, min(1, normalized))

    return (
        MIN_BUBBLE_DOMINANT_CH1
        + normalized ** 0.5
        * (MAX_BUBBLE_DOMINANT_CH1 - MIN_BUBBLE_DOMINANT_CH1)
    )


dominant_accord_pdf_ch1["bubble_size"] = (
    dominant_counts_ch1.apply(dominant_bubble_size_ch1)
)


# ------------------------------------------------------------
# 3. SELECT CATEGORIES TO LABEL
# ------------------------------------------------------------
# Label selection includes:
# - 8 largest categories by fragrance count;
# - 6 highest and 6 lowest categories by mean votes;
# - 6 highest and 6 lowest categories by mean rating.
#
# Duplicates are removed, largest categories receive priority,
# and the final number of labels is capped at 24.

largest_dominant_ch1 = (
    dominant_accord_pdf_ch1
    .nlargest(
        N_LARGEST_DOMINANT_CH1,
        "fragrance_count"
    )
)

dominant_labels_ch1 = pd.concat(
    [
        dominant_accord_pdf_ch1.nlargest(
            N_EXTREMES_DOMINANT_CH1,
            "mean_votes"
        ),

        dominant_accord_pdf_ch1.nsmallest(
            N_EXTREMES_DOMINANT_CH1,
            "mean_votes"
        ),

        dominant_accord_pdf_ch1.nlargest(
            N_EXTREMES_DOMINANT_CH1,
            "mean_rating"
        ),

        dominant_accord_pdf_ch1.nsmallest(
            N_EXTREMES_DOMINANT_CH1,
            "mean_rating"
        ),

        largest_dominant_ch1
    ]
).drop_duplicates("dominant_accord").copy()


dominant_labels_ch1["is_largest"] = (
    dominant_labels_ch1["dominant_accord"].isin(
        largest_dominant_ch1["dominant_accord"]
    )
)

dominant_labels_ch1["attention_extremeness"] = (
    (
        dominant_labels_ch1["mean_votes"]
        - popularity_median_dominant_ch1
    ).abs()
    / max(abs(popularity_median_dominant_ch1), 1)
)

dominant_labels_ch1["rating_extremeness"] = (
    dominant_labels_ch1["mean_rating"]
    - rating_median_dominant_ch1
).abs()

dominant_labels_ch1 = (
    dominant_labels_ch1
    .sort_values(
        [
            "is_largest",
            "attention_extremeness",
            "rating_extremeness"
        ],
        ascending=[False, False, False]
    )
    .head(MAX_LABELS_DOMINANT_CH1)
)


# ------------------------------------------------------------
# 4. DEFINE SEGMENT COLOURS
# ------------------------------------------------------------

segment_colors_dominant_ch1 = {
    "Standout Profiles": "#A66A4E",
    "Hidden Favourites": "#B58A9A",
    "Widely Explored": "#71818A",
    "Quiet Profiles": "#C5A85C"
}


# ------------------------------------------------------------
# 5. CREATE FIGURE
# ------------------------------------------------------------
# Match Figure 1.2 dimensions for consistent title rendering.

fig_1_3, ax = plt.subplots(
    figsize=(12, 10),
    dpi=180
)

fig_1_3.patch.set_facecolor(
    BOOK_STYLE["background"]
)

ax.set_facecolor(
    BOOK_STYLE["background"]
)


# ------------------------------------------------------------
# 6. PLOT ALL ELIGIBLE DOMINANT ACCORD CATEGORIES
# ------------------------------------------------------------

for segment_name in segment_order_dominant_ch1:

    subset = dominant_accord_pdf_ch1[
        dominant_accord_pdf_ch1["segment"] == segment_name
    ]

    if not subset.empty:

        ax.scatter(
            subset["mean_votes"],
            subset["mean_rating"],
            s=subset["bubble_size"],
            color=segment_colors_dominant_ch1[segment_name],
            alpha=0.78,
            edgecolors=BOOK_STYLE["background"],
            linewidths=0.8,
            zorder=3
        )


# ------------------------------------------------------------
# 7. ADD ACCORD LABELS
# ------------------------------------------------------------
# Labels show accord name and number of fragrances.
# Background is semi-transparent and has no visible border.

texts = []

for _, row in dominant_labels_ch1.iterrows():

    label = (
        f"{row['dominant_accord'].title()} "
        f"(n={row['fragrance_count']:,.0f})"
    )

    text = ax.text(
        row["mean_votes"],
        row["mean_rating"],
        label,
        color=BOOK_STYLE["text"],
        fontsize=BOOK_SIZES["category_pt"] + 1,
        fontfamily=BOOK_TYPOGRAPHY["body"],
        fontweight="bold",
        alpha=0.98,
        zorder=5,
        bbox={
            "facecolor": BOOK_STYLE["background"],
            "edgecolor": "none",
            "alpha": 0.85,
            "pad": 1.5
        }
    )

    texts.append(text)


# ------------------------------------------------------------
# 8. REDUCE LABEL OVERLAP
# ------------------------------------------------------------

if texts:

    adjust_text(
        texts,
        ax=ax,
        expand=(1.2, 1.4),
        force_text=(0.5, 0.8),
        force_static=(0.5, 0.8),
        force_pull=(0.05, 0.1),
        ensure_inside_axes=True,
        iter_lim=500,
        arrowprops=dict(
            arrowstyle="-",
            color=BOOK_STYLE["muted"],
            lw=0.7,
            alpha=0.7
        )
    )


# ------------------------------------------------------------
# 9. ADD MEDIAN REFERENCE LINES
# ------------------------------------------------------------

ax.axvline(
    popularity_median_dominant_ch1,
    color=BOOK_STYLE["muted"],
    linestyle="--",
    linewidth=1.1,
    alpha=0.9,
    zorder=2
)

ax.axhline(
    rating_median_dominant_ch1,
    color=BOOK_STYLE["muted"],
    linestyle="--",
    linewidth=1.1,
    alpha=0.9,
    zorder=2
)


# ------------------------------------------------------------
# 10. FORMAT AXES AND GRID
# ------------------------------------------------------------

ax.set_xlabel(
    "Mean community votes per fragrance",
    fontsize=BOOK_SIZES["category_pt"] + 1,
    fontfamily=BOOK_TYPOGRAPHY["body"],
    color=BOOK_STYLE["muted"],
    labelpad=12
)

ax.set_ylabel(
    "Mean fragrance rating",
    fontsize=BOOK_SIZES["category_pt"] + 1,
    fontfamily=BOOK_TYPOGRAPHY["body"],
    color=BOOK_STYLE["muted"],
    labelpad=12
)

ax.xaxis.set_major_formatter(
    mtick.FuncFormatter(
        lambda x, _: f"{x:,.0f}"
    )
)

ax.yaxis.set_major_formatter(
    mtick.FuncFormatter(
        lambda y, _: f"{y:.2f}"
    )
)

ax.tick_params(
    axis="both",
    length=0,
    labelsize=BOOK_SIZES["footer_pt"] + 1,
    pad=8,
    colors=BOOK_STYLE["text"]
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():

    tick.set_fontfamily(
        BOOK_TYPOGRAPHY["body"]
    )

ax.grid(
    axis="both",
    color=BOOK_STYLE["grid"],
    linewidth=0.7,
    alpha=0.9,
    zorder=0
)

ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)


# ------------------------------------------------------------
# 11. LEGEND — RELATIVE SEGMENTS
# ------------------------------------------------------------
# The segment legend is now positioned at the former
# bubble-size legend location.

segment_handles_dominant_ch1 = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        markersize=9,
        markerfacecolor=segment_colors_dominant_ch1[name],
        markeredgecolor=BOOK_STYLE["background"],
        markeredgewidth=0.8,
        label=name
    )
    for name in segment_order_dominant_ch1
]

segment_legend_dominant_ch1 = ax.legend(
    handles=segment_handles_dominant_ch1,
    title="Relative segment",
    loc="lower right",
    bbox_to_anchor=(1.0, 0.0),
    frameon=True,
    facecolor=BOOK_STYLE["background"],
    edgecolor=BOOK_STYLE["rule"],
    fontsize=BOOK_SIZES["category_pt"] + 1,
    title_fontsize=BOOK_SIZES["category_pt"] + 2,
    labelcolor=BOOK_STYLE["text"],
    borderpad=0.9,
    labelspacing=0.75,
    handletextpad=0.8
)

segment_legend_dominant_ch1.get_title().set_color(
    BOOK_STYLE["muted"]
)

segment_legend_dominant_ch1.get_title().set_fontfamily(
    BOOK_TYPOGRAPHY["body"]
)


# ------------------------------------------------------------
# 12. ADD FIGURE HEADER
# ------------------------------------------------------------

add_book_figure_header(
    fig_1_3,
    "Dominant Accord Segmentation",
    "All eligible dominant accord categories  |  "
    "Labels show accord and fragrance count."
)


# ------------------------------------------------------------
# 13. APPLY FIGURE LAYOUT
# ------------------------------------------------------------

fig_1_3.subplots_adjust(
    left=0.085,
    right=0.985,
    top=1 - BOOK_FIGURE_CHART_TOP_IN / fig_1_3.get_figheight(),
    bottom=0.11
)


# ------------------------------------------------------------
# 14. RENDER FIGURE
# ------------------------------------------------------------

show_book_matplotlib(fig_1_3)


# ------------------------------------------------------------
# 15. VALIDATION SUMMARY
# ------------------------------------------------------------

print(
    f"Eligible dominant accord categories: "
    f"{len(dominant_accord_pdf_ch1):,}"
)

print(
    f"Categories labelled: "
    f"{len(dominant_labels_ch1):,}"
)

print(
    f"Attention median: "
    f"{popularity_median_dominant_ch1:.2f}"
)

print(
    f"Rating median: "
    f"{rating_median_dominant_ch1:.3f}"
)

## 1.6 Figures 1.4–1.7 — Individual Dominant-Accord Segment Profiles

Each chart includes every dominant-accord category assigned to the relevant segment. Bar length represents mean community votes per fragrance; the data label also reports the mean fragrance rating. Segment membership is determined by the median thresholds, not by bar order.

In [0]:

# ============================================================
# FIGURES 1.4–1.7 — ALL DOMINANT ACCORDS WITHIN EACH SEGMENT
# Consistent sizing and spacing with Figures 1.1–1.3.
# ============================================================

def plot_dominant_segment_ch1(segment_name, figure_number):

    # --------------------------------------------------------
    # Prepare segment data
    # --------------------------------------------------------

    plot_df = (
        dominant_accord_pdf_ch1[
            dominant_accord_pdf_ch1["segment"] == segment_name
        ]
        .sort_values(
            ["mean_votes", "dominant_accord"],
            ascending=[True, True]
        )
        .copy()
    )

    if plot_df.empty:
        print(f"No dominant accords found for segment: {segment_name}")
        return None

    plot_df["display_label"] = plot_df["dominant_accord"].str.title()

    # Consistent compressed category spacing.
    y_positions = np.arange(len(plot_df)) * 0.80

    # Dynamic figure height, while retaining the same 12-inch width.
    fig_height = max(7.5, len(plot_df) * 0.52 + 2.5)

    # --------------------------------------------------------
    # Create chart
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(12, fig_height),
        dpi=180
    )

    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])

    bars = ax.barh(
        y_positions,
        plot_df["mean_votes"],
        height=0.462,
        color=segment_colors_dominant_ch1[segment_name],
        edgecolor="none",
        zorder=3
    )

    # --------------------------------------------------------
    # Category labels
    # --------------------------------------------------------

    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        plot_df["display_label"],
        fontsize=BOOK_SIZES["category_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        color=BOOK_STYLE["text"]
    )

    # Set limits around the compressed category positions.
    ax.set_ylim(
        y_positions[0] - 0.5,
        y_positions[-1] + 0.5
    )

    # --------------------------------------------------------
    # Data labels
    # Mean votes and mean rating.
    # --------------------------------------------------------

    max_value = float(plot_df["mean_votes"].max())
    label_offset = max(max_value * 0.022, 0.5)

    for bar, (_, row) in zip(bars, plot_df.iterrows()):

        ax.text(
            bar.get_width() + label_offset,
            bar.get_y() + bar.get_height() / 2,
            (
                f"{row['mean_votes']:,.1f} votes"
                f"  |  rating {row['mean_rating']:.3f}"
            ),
            va="center",
            ha="left",
            fontsize=BOOK_SIZES["value_pt"],
            fontfamily=BOOK_TYPOGRAPHY["body"],
            color=BOOK_STYLE["text"]
        )

    # Reserve enough horizontal space for the data labels.
    ax.set_xlim(
        0,
        max_value * 1.55 if max_value > 0 else 1
    )

    # --------------------------------------------------------
    # Axes
    # --------------------------------------------------------

    ax.set_xlabel(
        "Mean community votes per fragrance",
        fontsize=BOOK_SIZES["category_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        color=BOOK_STYLE["muted"],
        labelpad=10
    )

    ax.set_ylabel("")

    ax.xaxis.set_major_formatter(
        mtick.StrMethodFormatter("{x:,.0f}")
    )

    ax.tick_params(
        axis="y",
        length=0,
        labelsize=BOOK_SIZES["category_pt"],
        pad=10,
        colors=BOOK_STYLE["text"]
    )

    ax.tick_params(
        axis="x",
        length=0,
        labelsize=BOOK_SIZES["footer_pt"],
        pad=8,
        colors=BOOK_STYLE["muted"]
    )

    for tick in ax.get_yticklabels():
        tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])
        tick.set_fontsize(BOOK_SIZES["category_pt"])
        tick.set_color(BOOK_STYLE["text"])

    for tick in ax.get_xticklabels():
        tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])

    # --------------------------------------------------------
    # Grid and frame
    # --------------------------------------------------------

    ax.grid(
        axis="x",
        color=BOOK_STYLE["grid"],
        linewidth=0.8,
        alpha=0.9,
        zorder=0
    )

    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_visible(False)

    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------

    add_book_figure_header(
        fig,
        segment_name,
        "All dominant-accord categories  |  Bar length: mean votes per fragrance  |  Labels: mean rating"
    )

    # --------------------------------------------------------
    # Layout
    # Same margins and header-to-chart adjustment as Figure 1.1.
    # --------------------------------------------------------

    apply_book_bar_layout(
        fig,
        bottom=0.10,
        right=0.85
    )

    # Tighten the gap between the header and chart body.
    ax_pos = ax.get_position()

    ax.set_position([
        ax_pos.x0,
        ax_pos.y0,
        ax_pos.width,
        ax_pos.height + 0.035
    ])

    # --------------------------------------------------------
    # Save and render
    # --------------------------------------------------------

    filename = (
        f"figure_1_{figure_number}_"
        f"{segment_name.lower().replace(' ', '_')}.png"
    )

    output_path = CHAPTER_1_OUTPUT_DIR / filename

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches=None,
        facecolor=BOOK_STYLE["background"],
        edgecolor=BOOK_STYLE["background"]
    )

    show_book_matplotlib(fig)

    print(f"Saved: {output_path}")

    return fig


# ============================================================
# GENERATE FIGURES 1.4–1.7
# ============================================================

fig_1_4 = plot_dominant_segment_ch1("Standout Profiles", 4)
fig_1_5 = plot_dominant_segment_ch1("Hidden Favourites", 5)
fig_1_6 = plot_dominant_segment_ch1("Widely Explored", 6)
fig_1_7 = plot_dominant_segment_ch1("Quiet Profiles", 7)


# ============================================================
# VALIDATION
# ============================================================

print("\nDominant-accord segment figure summary:")

for segment_name in segment_order_dominant_ch1:

    segment_df = dominant_accord_pdf_ch1[
        dominant_accord_pdf_ch1["segment"] == segment_name
    ]

    print(
        f"{segment_name}: "
        f"{len(segment_df):,} categories"
    )

## 1.7 Export Chapter 1 figures

Export the seven reviewed figures as high-resolution PNG files and scalable SVG masters to the shared visualisations directory. Run this cell after reviewing the figures.

In [0]:
# ============================================================
# CHAPTER 1 FIGURE EXPORT — PNG + SVG
# ============================================================

chapter_1_figures_ch1 = [
    (fig_1_1, "figure_1_1_overall_accord_prevalence"),
    (fig_1_2, "figure_1_2_dominant_accord_distribution"),
    (fig_1_3, "figure_1_3_dominant_accord_segmentation"),
    (fig_1_4, "figure_1_4_standout_profiles"),
    (fig_1_5, "figure_1_5_hidden_favourites"),
    (fig_1_6, "figure_1_6_widely_explored"),
    (fig_1_7, "figure_1_7_quiet_profiles")
]

saved_chapter_1_files_ch1 = []
for figure, stem in chapter_1_figures_ch1:
    if figure is None:
        continue

    png_path = CHAPTER_1_OUTPUT_DIR / f"{stem}.png"
    svg_path = CHAPTER_1_OUTPUT_DIR / f"{stem}.svg"

    figure.savefig(
        png_path, format="png", dpi=400,
        bbox_inches="tight", facecolor=BOOK_STYLE["background"]
    )
    figure.savefig(
        svg_path, format="svg",
        bbox_inches="tight", facecolor=BOOK_STYLE["background"]
    )
    saved_chapter_1_files_ch1.extend([png_path, svg_path])

missing_chapter_1_files_ch1 = [
    str(path) for path in saved_chapter_1_files_ch1 if not path.is_file()
]
assert not missing_chapter_1_files_ch1, (
    f"Chapter 1 export verification failed: {missing_chapter_1_files_ch1}"
)

print(f"Chapter 1 figures exported: {len(saved_chapter_1_files_ch1)} files")
for path in saved_chapter_1_files_ch1:
    print(path)

## Reproducibility notes

- Source table and accord parsing are reused from Notebook 14.
- Overall prevalence counts each unique fragrance–accord association once.
- Dominant accord is extracted from the first position of the parsed accord array.
- The distribution figure includes all records with a valid first-listed accord.
- Attention and rating aggregation uses records with a non-null rating and a non-negative community rating count.
- Median segmentation is applied to dominant-accord categories with at least 100 fragrances in the rated analytical population.
- Categories equal to a median are assigned to the above-median side, consistent with Notebook 14's dyad and triad segmentation.
- The bubble chart plots all eligible categories but labels a selected subset; the four segment bar charts show every eligible category.
- Community votes represent platform activity, not sales, market share or unique consumers.

**Output location:** /Volumes/main/fragrance_analytics/reference_data/visualizations/